# Insight 3: Predict Client Acquisition Cost Payback Period

**Algorithm:** Ridge Regression (Linear Regression + L2 penalty),
implemented completely from scratch using batch gradient descent — no
scikit-learn.

**Business Question:** "How many months until this client becomes
profitable, given what it cost to acquire them (ads, referral bonus,
onboarding time)?"

**Why Ridge specifically:** `acquisition_channel_cost`, `onboarding_hours`,
`expected_annual_fee`, and `expected_retention_years` are all strongly
correlated with each other (costlier acquisition channels tend to bring
in clients who need more onboarding, who tend to be higher-fee clients,
who tend to stay longer). Plain linear regression can become unstable on
this kind of multicollinearity; Ridge's L2 penalty shrinks and stabilizes
the coefficients.

**Business use:** know which acquisition channels/clients pay back
fastest, so marketing and sales can prioritize spend on channels that
convert into profitable clients quickly.

> Run `1_generate_dataset.ipynb` first so `data/client_data.csv` exists.

## Shared helper functions

Manual implementations of train/test split, feature standardization, and
regression metrics. No scikit-learn is used — everything is plain NumPy.

In [1]:
import numpy as np
import pandas as pd

def train_test_split(X, y, test_size=0.2, seed=42):
    np.random.seed(seed)
    n = X.shape[0]
    indices = np.random.permutation(n)
    test_count = int(n * test_size)
    test_idx = indices[:test_count]
    train_idx = indices[test_count:]
    return X[train_idx], X[test_idx], y[train_idx], y[test_idx]

def standardize(X, mean=None, std=None):
    if mean is None:
        mean = X.mean(axis=0)
    if std is None:
        std = X.std(axis=0)
        std[std == 0] = 1.0
    return (X - mean) / std, mean, std

def r2_score(y_true, y_pred):
    ss_res = np.sum((y_true - y_pred) ** 2)
    ss_tot = np.sum((y_true - np.mean(y_true)) ** 2)
    return 1 - (ss_res / ss_tot)

def rmse(y_true, y_pred):
    return np.sqrt(np.mean((y_true - y_pred) ** 2))

def mae(y_true, y_pred):
    return np.mean(np.abs(y_true - y_pred))

def print_regression_report(y_true, y_pred, title="Regression Report"):
    print(f"\n--- {title} ---")
    print(f"R^2  : {r2_score(y_true, y_pred):.4f}")
    print(f"RMSE : {rmse(y_true, y_pred):.4f}")
    print(f"MAE  : {mae(y_true, y_pred):.4f}")

## The models: Ridge Regression (and a plain Linear baseline for comparison)

In [2]:
class RidgeRegressionScratch:
    """Linear regression with an L2 penalty on the weights (not the bias)."""

    def __init__(self, lr=0.05, n_iters=3000, alpha=1.0):
        self.lr = lr
        self.n_iters = n_iters
        self.alpha = alpha  # regularization strength (lambda)
        self.weights = None
        self.bias = None
        self.loss_history = []

    def fit(self, X, y):
        n_samples, n_features = X.shape
        self.weights = np.zeros(n_features)
        self.bias = 0.0

        for i in range(self.n_iters):
            y_pred = X @ self.weights + self.bias
            error = y_pred - y

            # MSE gradient + L2 penalty gradient (2*alpha*w), bias not penalized
            dw = (2 / n_samples) * (X.T @ error) + (2 * self.alpha / n_samples) * self.weights
            db = (2 / n_samples) * np.sum(error)

            self.weights -= self.lr * dw
            self.bias -= self.lr * db

            if i % 200 == 0 or i == self.n_iters - 1:
                mse = np.mean(error ** 2)
                penalty = self.alpha * np.sum(self.weights ** 2)
                self.loss_history.append(mse + penalty)

        return self

    def predict(self, X):
        return X @ self.weights + self.bias

## Load data and prepare features (deliberately correlated set)

In [4]:
df = pd.read_csv("client_data.csv")

feature_cols = [
    "acquisition_channel_cost",
    "onboarding_hours",
    "expected_annual_fee",
    "expected_retention_years",
]
target_col = "payback_period_months"

X = df[feature_cols].values.astype(float)
y = df[target_col].values.astype(float)

X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2)
X_train_s, mean, std = standardize(X_train)
X_test_s, _, _ = standardize(X_test, mean, std)

print("Feature correlation matrix:")
pd.DataFrame(X_train, columns=feature_cols).corr().round(2)

Feature correlation matrix:


,acquisition_channel_cost,onboarding_hours,expected_annual_fee,expected_retention_years
acquisition_channel_cost,1.00,0.96,0.94,0.74
onboarding_hours,0.96,1.00,0.98,0.77
expected_annual_fee,0.94,0.98,1.00,0.78
expected_retention_years,0.74,0.77,0.78,1.00


## Train: plain Linear Regression baseline (alpha = 0)

In [5]:
plain_model = RidgeRegressionScratch(lr=0.1, n_iters=4000, alpha=0.0)
plain_model.fit(X_train_s, y_train)
y_pred_plain = plain_model.predict(X_test_s)

print_regression_report(y_test, y_pred_plain, title="Plain Linear Regression (alpha=0) - TEST")


--- Plain Linear Regression (alpha=0) - TEST ---
R^2  : 0.9831
RMSE : 0.6172
MAE  : 0.4944


## Train: Ridge Regression (alpha = 1.0)

In [6]:
ridge = RidgeRegressionScratch(lr=0.1, n_iters=4000, alpha=1.0)
ridge.fit(X_train_s, y_train)
y_pred_ridge = ridge.predict(X_test_s)

print_regression_report(y_test, y_pred_ridge, title="Ridge Regression (alpha=1.0) - TEST")


--- Ridge Regression (alpha=1.0) - TEST ---
R^2  : 0.9831
RMSE : 0.6165
MAE  : 0.4938


## Compare coefficients (Ridge stabilizes weights vs plain Linear)

In [7]:
print("Coefficient comparison (Plain Linear vs Ridge):")
for name, w_plain, w_ridge in zip(feature_cols, plain_model.weights, ridge.weights):
    print(f"  {name:30s}  plain={w_plain:8.3f}   ridge={w_ridge:8.3f}")

Coefficient comparison (Plain Linear vs Ridge):
  acquisition_channel_cost        plain=   4.719   ridge=   4.696
  onboarding_hours                plain=   1.349   ridge=   1.338
  expected_annual_fee             plain=  -1.330   ridge=  -1.296
  expected_retention_years        plain=  -0.186   ridge=  -0.187


## Business output: fastest vs slowest payback clients

In [8]:
X_all_s, _, _ = standardize(X, mean, std)
df["predicted_payback_months"] = ridge.predict(X_all_s)

fastest = df.sort_values("predicted_payback_months", ascending=True).head(10)
print("Top 10 fastest-payback clients (best acquisition ROI):")
print(fastest[["client_id", "acquisition_channel_cost", "onboarding_hours",
                "expected_annual_fee", "expected_retention_years",
                "predicted_payback_months"]].round(2).to_string(index=False))

slowest = df.sort_values("predicted_payback_months", ascending=False).head(5)
print("\nTop 5 slowest-payback clients (worst acquisition ROI, review channel/pricing):")
print(slowest[["client_id", "acquisition_channel_cost", "onboarding_hours",
                "expected_annual_fee", "expected_retention_years",
                "predicted_payback_months"]].round(2).to_string(index=False))

avg_payback = df["predicted_payback_months"].mean()
print(f"\nAverage predicted payback period across all clients: {avg_payback:.2f} months")

Top 10 fastest-payback clients (best acquisition ROI):
 client_id  acquisition_channel_cost  onboarding_hours  expected_annual_fee  expected_retention_years  predicted_payback_months
       448                    811.37               1.1              3228.31                       3.9                      1.24
       289                    862.06               0.7              3235.83                       3.1                      1.37
        87                    858.05               1.7              3810.40                       2.9                      1.55
       768                    823.60               1.3              3438.41                       2.5                      1.61
      1892                    806.88               0.5              2783.81                       2.5                      1.62
        44                    813.21               1.4              3397.73                       2.6                      1.65
      2302                    824.49             